# ETL-09 · Perfil 2023

- **Etapa:** ETL-09, Perfil de 2023 
- **Responsável:** Pedro Wellington
- **Issue:** #ETL-09
- **Entrada:** `data/pessoas_*-2023.csv`
- **Saída:** `docs/perfil/perfil_2023.md`

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

DADOS  = RAIZ / "data"                      # CSVs brutos (estão no Git)

ENTRADA = DADOS      # pasta com os CSVs brutos
SAIDA   = RAIZ  / "docs" / "perfil" / "perfil_2023.md"

## 2. Carregar a entrada

In [2]:
arquivos = [DADOS / f"pessoas_{mes:02d}-2023.csv" for mes in range(1, 13)] # lista com os 12 endereços dos arquivos de 2023
df = pd.concat(
    [pd.read_csv(a, encoding='latin-1', sep=';') for a in arquivos],
    ignore_index=True                                                 # lê cada arquivo e empilha os 12 numa tabela só -> df
)
print('shape:', df.shape)
df.head()

shape: (279963, 31)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,dia_sinistro,ano_mes_sinistro,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito
0,1843599,1575881,735974.0,3520103,IGARAPAVA,FRANCA,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,1,2023/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN
1,1638444,1441234,2563314.0,3548500,SANTOS,BAIXADA SANTISTA,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,1,2023/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN
2,1638444,1441233,2558885.0,3548500,SANTOS,BAIXADA SANTISTA,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,1,2023/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN
3,1717503,676770,696266.0,3506359,BERTIOGA,BAIXADA SANTISTA,ESTRADAS E RODOVIAS,MOTOCICLETA,CONDUTOR,MOTOCICLETA,...,1,2023/01,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN
4,2044940,1836554,2829636.0,3523909,ITU,SOROCABA,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,1,2023/01,NaN,NaN,NaN,NaN,NaN,NaN,NAO DISPONIVEL,NaN


## 3. Diagnóstico ANTES

In [ ]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,142130
id_pessoa,int64,0.00,279963
id_veiculo,float64,21.77,190652
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,21.73,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,21.29,8


## 4. Cruzamento: veículo nulo × tipo de vítima

In [8]:
# linhas em que o tipo de veículo da vítima está vazio
sem_veiculo = df[df['tipo_veiculo_vitima'].isna()]

# dessas, quantas são de cada tipo de vítima (pedestre, condutor, passageiro...)
print(sem_veiculo['tipo_de_vitima'].value_counts(dropna=False))

# % de pedestres entre as que não têm veículo
print(round(len(sem_veiculo) / len(df) * 100, 2))

tipo_de_vitima
NAO DISPONIVEL    37072
PEDESTRE          15524
PASSAGEIRO         7083
CONDUTOR           1164
Name: count, dtype: int64
21.73


## 5. Óbitos

In [9]:
# quantas linhas têm data de óbito preenchida (= quem morreu)
obitos = df['data_obito'].notna().sum()
print('óbitos:', obitos, '| % do total:', round(obitos / len(df) * 100, 2))

# as 7 colunas de óbito são nulas juntas?
# True  = em toda linha, as 7 estão TODAS vazias (não morreu) ou TODAS preenchidas (morreu)
# False = alguma linha tem óbito preenchido pela metade (dado quebrado)
cols_obito = ['data_obito', 'ano_obito', 'mes_obito', 'dia_obito',
              'ano_mes_obito', 'local_obito', 'tempo_sinistro_obito']
print((df[cols_obito].isna().all(axis=1) | df[cols_obito].notna().all(axis=1)).all())

óbitos: 5490 | % do total: 1.96
True


## 6. Valores corrompidos (acentos quebrados)

In [13]:
# colunas de texto (o pandas novo guarda texto como 'str', por isso as duas opções)
colunas_texto = df.select_dtypes(include=['object', 'str']).columns

achados = []
for coluna in colunas_texto:
    # valores diferentes da coluna, sem os vazios
    valores = pd.Series(df[coluna].dropna().unique())

    # suspeitos: letra sozinha (acento perdido, ex.: "S O PAULO") ou "Ã" (acento mal lido)
    suspeitos = valores[valores.str.contains(r'\b[B-DF-NP-TV-Z]\b|Ã', regex=True)]

    # guarda cada suspeito com o nome da coluna onde apareceu
    for v in suspeitos:
        achados.append({'coluna': coluna, 'valor': v})

# transforma a lista numa tabela e mostra os 20 primeiros
corrompidos = pd.DataFrame(achados)
print(len(corrompidos), 'valores suspeitos')
corrompidos.head(20)
# todos os suspeitos, não só os 20 primeiros
corrompidos.tail(18)

# conta os caracteres fora do alfabeto simples (se só aparecerem Ã, É, Ç, Ê..., os acentos estão ok)
from collections import Counter
chars = Counter()
for coluna in colunas_texto:
    for v in df[coluna].dropna().unique():
        chars.update(c for c in v if ord(c) > 127)
        
# tira os valores com acento (estão certos) e fica só com os suspeitos de verdade
reais = corrompidos[corrompidos['valor'].map(str.isascii)]
print(len(reais), 'valores realmente suspeitos')
print(reais.to_string())

# salva a lista em arquivo
reais.to_csv(RAIZ / "docs" / "perfil" / "corrompidos_2023.csv", index=False)


38 valores suspeitos
8 valores realmente suspeitos
      coluna                   valor
0  municipio        PALMEIRA D OESTE
1  municipio   SANTA BARBARA D OESTE
2  municipio      SANTA RITA D OESTE
3  municipio       APARECIDA D OESTE
4  municipio         ESTRELA D OESTE
5  municipio     SANTA CLARA D OESTE
6  municipio  SAO JOAO DO PAU D ALHO
7  municipio         GUARANI D OESTE


## 7. Resumo

> O que foi feito, por quê, e quantas linhas/valores mudaram.
> Este texto vira a subseção do relatório (`docs/relatorio/secoes/`).